In [24]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, log_loss

# Data
cols = ["game_date", "game_pk", "game_type", "at_bat_number", "pitch_number",
        "pitcher", "batter", "pitch_type", "description",
        "release_speed", "release_spin_rate", "pfx_x", "pfx_z", "plate_x", "plate_z",
        "balls", "strikes", "stand", "p_throws",
        "release_pos_x", "release_pos_y", "release_pos_z",
        "vx0", "vy0", "vz0", "ax", "ay", "az"]

files = sorted(Path("data").glob("statcast_*.parquet"))
df = pd.concat([pd.read_parquet(f, columns=cols) for f in files], ignore_index=True)

WHIFF_DESCRIPTIONS = ["swinging_strike", "swinging_strike_blocked", "missed_bunt", "foul_tip"]

swing_descriptions = ["swinging_strike", "swinging_strike_blocked", "missed_bunt",
                      "foul", "foul_tip", "hit_into_play", "foul_bunt", "bunt_foul_tip"]

features = ["pitch_type", "release_speed", "release_spin_rate", "pfx_x", "pfx_z",
            "plate_x", "plate_z", "balls", "strikes", "stand", "p_throws"]

df["whiff"] = df["description"].isin(WHIFF_DESCRIPTIONS).astype(int)

In [25]:
df = df[df["game_type"] == "R"].copy()

In [26]:
print(df.shape)
print(df["game_type"].value_counts())
print(df["game_date"].min(), df["game_date"].max())
print(df.duplicated(["game_pk", "at_bat_number", "pitch_number"]).sum())   # want 0
print(df.groupby(df["game_date"].str[:7]).size()) 

(689901, 29)
game_type
R    689901
Name: count, dtype: int64
2026-04-01 2026-09-27
0
game_date
2026-04    117333
2026-05    121779
2026-06    115920
2026-07    105140
2026-08    123373
2026-09    106356
dtype: int64


In [27]:
# 1.) Sort Chronologically
df = df.sort_values(["game_pk", "at_bat_number", "pitch_number"]).reset_index(drop=True)

In [28]:
# 1a.) Build Sequence Features
group = ["game_pk", "at_bat_number"]
g = df.groupby(group)

df["prev_pitch_type"] = g["pitch_type"].shift(1)
df["prev_speed"] = g["release_speed"].shift(1)
df["speed_change"] = df["release_speed"] - df["prev_speed"]
df["plate_z_change"] = df["plate_z"] - g["plate_z"].shift(1)
df["plate_x_change"] = df["plate_x"] - g["plate_x"].shift(1)
df["prev_whiff"] = g["whiff"].shift(1)

df["prev2_pitch_type"] = g["pitch_type"].shift(2)

# same_type_as_prev column, 1 if its the same, 0 if its not, NaN if one is not available
df["same_type_as_prev"] = (
    (df["pitch_type"] == df["prev_pitch_type"]).astype(float)
        .where(df["prev_pitch_type"].notna())
)

cols = ["pitch_number", "pitch_type", "release_speed", "prev_pitch_type",
        "prev_speed", "speed_change", "same_type_as_prev", "description"]
print(df[(df["game_pk"] == df["game_pk"].iloc[0]) & (df["at_bat_number"] == 1)][cols])


   pitch_number pitch_type  release_speed prev_pitch_type  prev_speed  \
0             1         SI           92.7             NaN        <NA>   
1             2         CH           85.9              SI        92.7   
2             3         FF           93.1              CH        85.9   

   speed_change  same_type_as_prev    description  
0          <NA>                NaN  called_strike  
1          -6.8                0.0           ball  
2           7.2                0.0  hit_into_play  


In [29]:
# 2.) Filter to swings, Clean Baseline Data
swings = df[df["description"].isin(swing_descriptions)]
swings = swings.dropna(subset=features+["whiff"])

In [30]:
# 3.) Convert Text Columns to Categories
for col in ["pitch_type", "stand", "p_throws", "prev_pitch_type"]:
    swings[col] = swings[col].astype("category")

# 4.) Split By Date, not random
swings["game_date"] = pd.to_datetime(swings["game_date"])
cutoff = pd.Timestamp("2026-09-01")

train = swings[swings["game_date"] < cutoff]
test = swings[swings["game_date"] >= cutoff]

In [31]:
new_features = ["prev_pitch_type", "prev_speed", "speed_change", "same_type_as_prev",
                "plate_z_change", "plate_x_change", "prev_whiff"]
features_b = features + new_features

In [32]:
# 5.) Create and Fit Model A
model = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=42)
X_train = train[features]
y_train = train["whiff"]
X_test = test[features]
y_test = test["whiff"]

model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing val

In [33]:
# 5b.) Create and Fit Model B
model_b = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=42)
X_train_b = train[features_b]
X_test_b = test[features_b]

model_b.fit(X_train_b, y_train)

,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing val

In [34]:
seeds = [0, 1, 2, 3, 4]
rows = []
for seed in seeds:
    m_a = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=seed).fit(X_train, y_train)
    m_b = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=seed).fit(X_train_b, y_train)
    p_a = m_a.predict_proba(X_test)[:, 1]
    p_b = m_b.predict_proba(X_test_b)[:, 1]
    rows.append({"seed": seed,
                 "ll_a": log_loss(y_test, p_a), "ll_b": log_loss(y_test, p_b),
                 "auc_a": roc_auc_score(y_test, p_a), "auc_b": roc_auc_score(y_test, p_b)})

seed_results = pd.DataFrame(rows)
seed_results["ll_diff"] = seed_results["ll_b"] - seed_results["ll_a"]   # negative = B better
seed_results

,seed,ll_a,ll_b,auc_a,auc_b,ll_diff
0,0,0.463902,0.464218,0.768809,0.768250,0.000316
1,1,0.464523,0.464273,0.767754,0.768596,-0.000249
2,2,0.464239,0.463835,0.768082,0.769121,-0.000404
3,3,0.464496,0.463897,0.767823,0.768920,-0.000599
4,4,0.464141,0.464450,0.768300,0.768312,0.000309


In [35]:
# 6.) Score and Compare to Dumb Baseline
probs = model.predict_proba(X_test)[:,1]
baseline_probs = np.full(len(y_test), y_train.mean())
probs_b = model_b.predict_proba(X_test_b)[:,1]
results = {}

results["Baseline"] = {
    "auc": roc_auc_score(y_test, baseline_probs),
    "log_loss": log_loss(y_test, baseline_probs),
}
results["Model A"] = {
    "auc": roc_auc_score(y_test, probs),
    "log_loss": log_loss(y_test, probs),
}
results["Model B"] = {
    "auc": roc_auc_score(y_test, probs_b),
    "log_loss": log_loss(y_test, probs_b)
}

pd.DataFrame(results).T

print(f"Model Baseline AUC: {results["Baseline"]["auc"]}")
print(f"Model Baseline Log Loss: {results["Baseline"]["log_loss"]}")
print(f"Model Baseline Mean: {baseline_probs.mean()}\n")

print(f"Model A AUC: {results["Model A"]["auc"]}")
print(f"Model A Log Loss: {results["Model A"]["log_loss"]}")
print(f"Model A Mean: {probs.mean()}\n")

print(f"Model B AUC: {results["Model B"]["auc"]}")
print(f"Model B Log Loss: {results["Model B"]["log_loss"]}")
print(f"Model B Mean: {probs_b.mean()}\n")



Model Baseline AUC: 0.5
Model Baseline Log Loss: 0.5637259008915219
Model Baseline Mean: 0.24960243454626946

Model A AUC: 0.7677994020779519
Model A Log Loss: 0.4645320777472711
Model A Mean: 0.25017433118900745

Model B AUC: 0.7690528770612054
Model B Log Loss: 0.46385548661755993
Model B Mean: 0.2504689449217886



In [36]:
def per_swing_loss(y, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

d = per_swing_loss(y_test.values, probs_b) - per_swing_loss(y_test.values, probs)   # negative = B better

rng = np.random.default_rng(42)
boots = [rng.choice(d, size=len(d), replace=True).mean() for _ in range(2000)]
print(d.mean(), np.percentile(boots, [2.5, 97.5]))

-0.0006765911297112246 [-0.0012049  -0.00015623]


In [37]:
fastballs = ["FF", "SI", "FC"]
offspeed = ["SL", "ST", "CU", "KC", "CH", "FS"]

off = swings[swings["pitch_type"].isin(offspeed) & swings["prev_pitch_type"].notna()].copy()
off["after_fb"] = off["prev_pitch_type"].isin(fastballs)

off.groupby("after_fb")["whiff"].agg(["mean", "count"])
off.groupby(["pitch_type", "after_fb"])["whiff"].agg(["mean", "count"])
off.groupby("after_fb")[["strikes", "balls"]].mean()


,strikes,balls
after_fb,,
False,1.395521,1.109719
True,1.404186,1.117465


In [38]:
off.groupby(["strikes", "after_fb"])["whiff"].agg(["mean", "count"])

mean  count
strikes after_fb                 
0       False     0.371311   6811
        True      0.357962   5671
1       False     0.349561  22657
        True      0.305595  25092
2       False     0.311532  30549
        True      0.268602  30387

In [39]:
pp = (off.groupby(["pitcher", "pitch_type", "after_fb"])["whiff"]
         .agg(["mean", "count"])
         .unstack("after_fb"))
pp.columns = ["mean_no_fb", "mean_fb", "n_no_fb", "n_fb"]
pp = pp.dropna()
pp = pp[(pp["n_fb"] >= 10) & (pp["n_no_fb"] >= 10)]

pp["diff"] = pp["mean_fb"] - pp["mean_no_fb"]
pp["weight"] = pp["n_fb"] * pp["n_no_fb"] / (pp["n_fb"] + pp["n_no_fb"])

print(len(pp), np.average(pp["diff"], weights=pp["weight"]))

1100 -0.03846686166638573


In [40]:
rng = np.random.default_rng(42)
boots = []
for _ in range(2000):
    s = pp.sample(len(pp), replace=True, random_state=int(rng.integers(1_000_000_000)))
    boots.append(np.average(s["diff"], weights=s["weight"]))

print(np.percentile(boots, [2.5, 97.5]))
print((pp["n_fb"] + pp["n_no_fb"]).sum())

[-0.04445082 -0.03254106]
111302.0


In [41]:
t = test.copy()
t["pred_a"] = probs
t["pred_b"] = probs_b
t = t[t["pitch_type"].isin(offspeed) & t["prev_pitch_type"].notna()].copy()
t["after_fb"] = t["prev_pitch_type"].isin(fastballs)

print(len(t))
print(log_loss(t["whiff"], t["pred_a"]), log_loss(t["whiff"], t["pred_b"]))
t.groupby("after_fb")[["whiff", "pred_a", "pred_b"]].mean()

18473
0.4945993826854436 0.49326406991731037


,whiff,pred_a,pred_b
after_fb,,,
False,0.330251,0.318254,0.328786
True,0.296444,0.310084,0.298487


In [42]:
# Explore
print("\n-----Explore-----\n")
print(len(train), len(test))
print(len(train) / len(swings) * 100)
print(train["game_date"].max(), test["game_date"].min())
print(train["whiff"].mean(), test["whiff"].mean())

print(f"Number of Swings: {len(swings)}")
print(f"Number of Whiffs: {swings["whiff"].sum()}")

print(f"Whiff Rate per Swing: {swings["whiff"].mean() * 100}\n")

#print(swings[features].dtypes)

#print(df["whiff"].value_counts())
#print(df["whiff"].sum())
#print(df.head())

#print(f"\n{df[features].isna().sum()}\n")
#print(f"df.shape: {df.shape}")


-----Explore-----

276684 50867
84.47050993585732
2026-08-31 00:00:00 2026-09-01 00:00:00
0.2496024345462694 0.2512630978827137
Number of Swings: 327551
Number of Whiffs: 81842
Whiff Rate per Swing: 24.98603270941014

